In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Set)=
# Set

*Setting attribute values to a molecular system.*

Some attributes of a molecular system can be changed without altering its identity — for example, modifying box dimensions, coordinates, stable isotope mass numbers, or force field parameters. In other cases, attribute corrections may be required, such as fixing a group name or atom type. MolSysMT provides the {func}`molsysmt.basic.set` function to assist with these tasks.

:::{hint}
:class: dropdown

Visit the section {ref}`Introduction_Attributes` for a detailed overview of attributes in MolSysMT.
:::

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.set`.
:::


## Basic usage

Let's show how this function works using a PDB structure (`181L`):


In [2]:
import molsysmt as msm

In [3]:
molsys = msm.convert('181L')

In [4]:
msm.info(molsys)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_ions,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1441,302,141,6,141,5,136,2,2,1,1


Suppose that the name of the group at index 30 is incorrect in our system:

In [5]:
msm.info(molsys, element='atom', selection='group_index==30')

index,id,name,type,group index,group id,group name,group type,component index,chain index,molecule index,molecule type,entity index,entity name
248,249,N,N,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
249,250,CA,C,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
250,251,C,C,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
251,252,O,O,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
252,253,CB,C,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
253,254,CG,C,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
254,255,ND1,N,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
255,256,CD2,C,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
256,257,CE1,C,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME
257,258,NE2,N,30,31,HIS,amino acid,0,0,0,protein,0,T4 LYSOZYME


Let’s now assume that the histidine residue should be labeled as "HSD" instead of "HIS". This can be easily updated using the function {func}`molsysmt.basic.set`:

In [6]:
msm.set(molsys, selection='group_index==30', group_name='HSD')

:::{tip}
:class: dropdown

All methods defined in the {ref}`molsysmt.basic <API basic>` module can also be invoked from the main level of the library. Hence, {func}`molsysmt.set` is the same function as {func}`molsysmt.basic.set`.
:::


Let's check that the group name changed in deed:

In [7]:
msm.info(molsys, element='atom', selection='group_index==30')

index,id,name,type,group index,group id,group name,group type,component index,chain index,molecule index,molecule type,entity index,entity name
248,249,N,N,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
249,250,CA,C,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
250,251,C,C,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
251,252,O,O,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
252,253,CB,C,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
253,254,CG,C,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
254,255,ND1,N,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
255,256,CD2,C,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
256,257,CE1,C,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME
257,258,NE2,N,30,31,HSD,amino acid,0,0,0,protein,0,T4 LYSOZYME


Let’s explore another example, this time modifying a structural attribute. We will look at the coordinates of the "CA" atom in the "HSD" group:

In [8]:
msm.get(molsys, element='atom', selection='group_name=="HSD" and atom_name=="CA"', coordinates=True)

Magnitude,[[[3.7278 1.6327 1.7646999999999997]]]
Units,nanometer


There is only a single structure in the system, let's change the coordinates of this former atom to a new value:

In [9]:
msm.set(molsys, selection='group_name=="HSD" and atom_name=="CA"', structure_indices=0, coordinates='[0,0,0] nm')

Finally, let’s verify that the coordinates of the "CA" atom in the "HSD" group were updated correctly:

In [10]:
msm.get(molsys, element='atom', selection='group_name=="HSD" and atom_name=="CA"', coordinates=True)

Magnitude,[[[0.0 0.0 0.0]]]
Units,nanometer


## Initializing periodic boxes

When a native system has coordinates but no box series, a full-axis
`msm.set(molsys, box=box_quantity)` assignment initializes it. Supply matrices
with shape (n_structures, 3, 3) and explicit length units. Internal storage uses
nm; your input length units may differ. Partial updates require an existing
series and raise an error before initialization. They do not invent boxes for
unspecified structures. Periodic analyses use the assigned row lattice vectors;
see the executed {ref}`Tutorial_Get_pi_pi_interactions` example.


## Updating geometry with saved analyses

For a native MolSys with named `interactions`, coordinate or box writes through `msm.set(molsys, ...)` remove old observations and mark the selected structures unevaluated in every analysis. Untouched structures remain available, and old query views remain historical snapshots. This includes evaluated-empty structures: their old zero count cannot describe new geometry. Recalculate and attach new results explicitly.

Empty atom/frame selections and identifier/time changes do not invalidate. Full geometry assignment preserves the frame axis when analyses are attached; change the axis with extraction or append operations. A failure after geometry delegation starts conservatively leaves selected frames unevaluated. Editing a separately accessed Structures object requires explicit owner invalidation. See {doc}`the sparse interaction recipe <../../cookbook/querying_sparse_interactions>`.


Chemical-state atom and scientific bond edits with `msm.set(molsys, ...)`
invalidate every covered structure in the named analyses. Replacing
`molsys.chemical_states` does the same. Changing
`structure_chemical_state_index` invalidates only the selected frames.
Bond IDs and empty selections preserve results. No detector runs
automatically. Separate Topology/ChemicalStates aliases, topology
replacement, mechanics changes and raw data writes need explicit owner
invalidation before new calculations.


## Assigning mechanical charges

For a native MolSys, `msm.set(molsys, partial_charge=values)` writes mechanical
charges independently of formal charges. Supply a finite one-dimensional charge
quantity or numerical values in elementary charge, aligned with the selected
atom indices in their explicit order. Session charge units do not change native
storage. Unknown unselected values stay missing; `partial_charge=None` clears
the full column. An empty selection changes nothing. Explicit replacement clears
original charge-model attribution. See {ref}`Tutorial_Partial_Charge_Assignment`
for a named calculation instead of manual values.


## Setting mechanical labels

Mechanical atom labels are parameter assignments, separate from element symbols.
For a native MolSys or MolecularMechanics object, `msm.set(..., element='atom',
atom_ff_type=...)` accepts a one-dimensional string vector. Subset values follow
the requested index order. MolSys allocates its complete atom axis on the first
subset edit, leaving other labels unknown; standalone MolecularMechanics requires
an existing axis. Invalid lengths fail before mutation. An empty selection is a
no-op, and `atom_ff_type=None` clears only the full label column. Charges, chemical
assignments and coordinates remain intact. Manual labels are a caller declaration,
and replacement clears named typing provenance; it does not certify a chemical
scheme. Explicitly invalidate dependent analyses after mechanical edits.
See {ref}`Tutorial_Assign_AutoDock_Types` for calculated assignments and export.


## Updating atom labels

Native atom IDs are string labels, independently of atom indices. Integer labels are normalized to strings; lexical labels such as `00101` retain their zeros. A scalar label broadcasts over selected atoms. Use an aligned list for different labels. This example starts from a fresh bundled system.


In [11]:
import molsysmt as msm
molsys = msm.convert(msm.systems['T4 lysozyme L99A']['181l.pdb'], to_form='molsysmt.MolSys')
molsys = msm.extract(molsys, selection=[0, 1])
msm.set(molsys, selection=[0], atom_id=101)
msm.set(molsys, selection=[1], atom_id='00101')
labels = msm.get(molsys, element='atom', atom_id=True)
print(labels)
assert labels == ['101', '00101']
assert msm.get(molsys, element='atom', atom_index=True) == [0, 1]


['101', '00101']


## Assigning stored chemistry

`msm.set()` also accepts a standalone `ChemicalStates` collection. Atom/bond
positions belong to the requested chemical state; integer positions are unique
and values follow their order. None assigns unknown values, not a neutral charge.
Scientific assignments through `molsys.chemical_states` invalidate that owner's
named analyses; a bond-ID edit preserves observations. Each attribute is staged
individually, so a multi-attribute call is not one transaction. This does not run
chemical preparation. Assignments on the `ChemicalStatesDict` form remain pending. See the
{ref}`native chemical editing guide <Tutorial_Native_Chemical_Editing>`.


In [12]:
from molsysmt.native import ChemicalStates

chemical_states = ChemicalStates(n_atoms=3)
for _ in range(2):
    chemical_states.append_state()
msm.set(chemical_states, selection=[2, 0], chemical_state=1, formal_charge=[-1, 1])
print(chemical_states.get(chemical_state=1, formal_charge=True))
msm.set(chemical_states, selection=[0], chemical_state=1, formal_charge=None)
print(chemical_states.get(chemical_state=1, formal_charge=True))
assert chemical_states.get(chemical_state=0, formal_charge=True) is None
assert chemical_states.reference_chemical_state_index == 0


[1, None, -1]
[None, None, -1]


## Declaring state components

Component IDs, names and types are writable labels, with scalars or one-item
vectors broadcasting over selected rows. An atom projection updates its entire
component row and rejects unknown membership or conflicting labels from atoms
of one component. `component_index` declares membership in existing rows; a changed
declaration retains labels only for exact full atom sets and becomes partial/
user_defined. None means unknown membership. Explicit reconstruction evaluates the
stored graph; assignment does not infer components or stable molecules.


In [13]:
chemical_states.add_bonds([[0, 1]], chemical_state=1)
msm.set(chemical_states, chemical_state=1, component_id=['pair', 'single'])
msm.set(chemical_states, selection=[1], chemical_state=1, component_index=None)
print(chemical_states.get(element='atom', chemical_state=1, component_index=True))
print(chemical_states.get(element='component', chemical_state=1, component_id=True))
assert chemical_states.get(chemical_state=1, component_evidence=True) == 'user_defined'


[0, None, 1]
[None, 'single']


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Introduction_Attributes`: Overview of topological and structural attributes defined for molecular systems in MolSysMT.
- {ref}`Tutorial_Convert`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Tutorial_Info`: Display structural and topological summaries of a molecular system with {func}`molsysmt.basic.info`.
- {ref}`Tutorial_Set`: Set attribute values to a molecular system with {func}`molsysmt.basic.set`.
- {ref}`Tutorial_Get`: Retrieve attribute values from selected elements with {func}`molsysmt.basic.get`.
- {ref}`Tutorial_Select`: Select elements from a molecular system using selection syntax with {func}`molsysmt.basic.select`.
:::
